# Stage 1: run_validity_smoke
Run cells in order. Mount the intended Drive account. This notebook calls shared Python modules under `src/stage1/`; it does not require the removed Stage 1 `.py` entry scripts. Failure or blocked status stops the notebook. Keep the checkout clean; outputs belong under PHASE1_ROOT, never the repository. Actual Drive sync and model readiness require their own checks.
Run only after validate_model passes for the selected model and current configuration. Its FIT-only toy readout is a plumbing check, not scientific evidence or full G0. On one L4, run the two models sequentially.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path('/content/Research')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO_URL = 'https://github.com/Drazr/Research.git'
BRANCH = 'phase1'
STORAGE_SCOPE = 'persistent-drive'
DATASET_MANIFEST = None
PREPARATION_COMMIT = None

MODEL = 'qwen'  # Change to 'gemma' for its separate run.
DATASET_MANIFEST = Path('PASTE_CPU_NOTEBOOK_DATASET_MANIFEST')
MODEL_CONFIG = Path('PASTE_CONFIGURE_L4_MODELS_CONFIG_FOR_THIS_MODEL')


## Mount Drive and clone phase1


In [ ]:
try:
    from google.colab import drive
except ImportError:
    if STORAGE_SCOPE != 'local-development':
        raise RuntimeError('Outside Colab: configure an explicit local root and local-development scope.')
else:
    drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive mount is unavailable.')

if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Repository destination is nonempty; choose an empty path.')
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)

def git_output(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if git_output('remote', 'get-url', 'origin').removesuffix('.git').rstrip('/') != REPO_URL.removesuffix('.git'):
    raise RuntimeError('Expected the Drazr/Research repository.')
if git_output('branch', '--show-current') != BRANCH:
    raise RuntimeError('Select the phase1 branch before continuing.')
if git_output('status', '--porcelain'):
    raise RuntimeError('Use a clean checkout so the recorded code commit matches the scripts.')
CODE_COMMIT = git_output('rev-parse', 'HEAD')
PHASE1_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['PHASE1_ROOT'] = str(PHASE1_ROOT)
sys.path.insert(0, str(REPO))
print('Code commit:', CODE_COMMIT)
print('Storage root:', PHASE1_ROOT)


## Install pinned L4 requirements and define runner


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-l4.txt')], check=True)

COMMON_ARGS = [
    '--phase1-root', str(PHASE1_ROOT),
    '--code-commit', CODE_COMMIT,
    '--storage-scope', STORAGE_SCOPE,
]

def run_stage1_task(name, extra_args=()):
    if git_output('rev-parse', 'HEAD') != CODE_COMMIT or git_output('status', '--porcelain'):
        raise RuntimeError('Repository changed after setup; rerun setup from a clean checkout.')
    task_args = list(COMMON_ARGS)
    if name == 'validate_model.py':
        index = task_args.index('--code-commit')
        del task_args[index:index+2]
    command = [sys.executable, '-m', 'src.stage1.' + name.removesuffix('.py'), *task_args, *extra_args]
    report = None
    with subprocess.Popen(command, cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                candidate = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict) and 'status' in candidate:
                report = candidate
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'{name} failed (exit {returncode}); fix the error before continuing.')
    if report is None or report.get('status') != 'passed':
        raise RuntimeError(f'{name} did not return a passed report.')
    return report


## Model access
If needed, authenticate using the runtime secret HF_TOKEN or Hugging Face login. Do not paste tokens into notebook cells or saved outputs. Accept any required Gemma access terms through your own account.


In [ ]:
# Each Colab runtime needs its own authenticated Hugging Face session.
if MODEL == 'gemma':
    import os
    from getpass import getpass
    from huggingface_hub import login
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass('Hugging Face read token for your Gemma-authorized account: ')
    os.environ['HF_TOKEN'] = token
    login(token=token, add_to_git_credential=False)
    del token


## Run run_validity_smoke


In [ ]:
if MODEL not in ('qwen', 'gemma'):
    raise ValueError('MODEL must be qwen or gemma')
if not DATASET_MANIFEST.is_file():
    raise RuntimeError('Set DATASET_MANIFEST to the CPU notebook completion manifest.')
extra_args = ['--model', MODEL, '--dataset-manifest', str(DATASET_MANIFEST)]
if not MODEL_CONFIG.is_file():
    raise RuntimeError('Set MODEL_CONFIG to the successful configure_l4 output for this model.')
from src.common.storage import verify
configuration = verify(MODEL_CONFIG.parent, expected={'kind': 'stage1_runtime_configuration'})
if configuration['code_commit'] != CODE_COMMIT:
    raise RuntimeError('Configuration was produced by another code commit; rerun configure_l4.')
registry = json.loads(MODEL_CONFIG.read_text(encoding='utf-8'))
if registry['models'][MODEL]['config_fingerprint'] != configuration['model_fingerprint']:
    raise RuntimeError('Configuration artifact belongs to a different model.')
extra_args += ['--models-config', str(MODEL_CONFIG)]
extra_args += ['--batch-size', '1']
result = run_stage1_task('run_validity_smoke.py', extra_args)
